In [1]:
import wikipediaapi

In [2]:
import wikipediaapi
wiki_wiki = wikipediaapi.Wikipedia(user_agent='MyProjectName (merlin@example.com)', language='en')

page_py = wiki_wiki.page('Python_(programming_language)')

In [12]:
page_py = wiki_wiki.page('Java (Programming language)')
print("Page - Exists: %s" % page_py.exists())
# Page - Exists: True

page_missing = wiki_wiki.page('NonExistingPageWithStrangeName')
print("Page - Exists: %s" %     page_missing.exists())
# Page - Exists: False

Page - Exists: True
Page - Exists: False


In [14]:
import wikipediaapi
wiki_wiki = wikipediaapi.Wikipedia('MyProjectName (merlin@example.com)', 'en')

print("Page - Title: %s" % page_py.title)
# Page - Title: Python (programming language)

print("Page - Summary: %s" % page_py.summary[0:600])
# Page - Summary: Python is a widely used high-level programming language for

Page - Title: Java (programming language)
Page - Summary: Java is a high-level, general-purpose, memory-safe, object-oriented programming language. It is intended to let programmers write once, run anywhere (WORA), meaning that compiled Java code can run on all platforms that support Java without the need to recompile. Java applications are typically compiled to bytecode that can run on any Java virtual machine (JVM) regardless of the underlying computer architecture. The syntax of Java is similar to C and C++, but has fewer low-level facilities than either of them. The Java runtime provides dynamic capabilities (such as reflection and runtime code m


In [5]:
wiki_wiki = wikipediaapi.Wikipedia(
    user_agent='MyProjectName (merlin@example.com)',
    language='en',
    extract_format=wikipediaapi.ExtractFormat.WIKI
)

p_wiki = wiki_wiki.page("Test 1")
print(p_wiki.text)
# Summary
# Section 1
# Text of section 1
# Section 1.1
# Text of section 1.1
# ...


wiki_html = wikipediaapi.Wikipedia(
    user_agent='MyProjectName (merlin@example.com)',
    language='en',
    extract_format=wikipediaapi.ExtractFormat.HTML
)
p_html = wiki_html.page("Test 1")
print(p_html.text)
# <p>Summary</p>
# <h2>Section 1</h2>
# <p>Text of section 1</p>
# <h3>Section 1.1</h3>
# <p>Text of section 1.1</p>
# ...

In [6]:
def print_sections(sections, level=0):
    for s in sections:
        print("%s: %s - %s" % ("*" * (level + 1), s.title, s.text[0:40]))
        print_sections(s.sections, level + 1)

In [1]:
import wikipediaapi

# Initialize Wikipedia API object
wiki_wiki = wikipediaapi.Wikipedia(
    user_agent='MyProjectName (merlin@example.com)',  # Replace with your project info
    language='en',  # Language code (e.g., 'en' for English)
    extract_format=wikipediaapi.ExtractFormat.WIKI  # Format as wiki-markup
)

# Fetch a Wikipedia page (for example, "Artificial intelligence")
def fetch_wikipedia_page(title):
    page = wiki_wiki.page(title)
    
    # Check if the page exists
    if not page.exists():
        return f"Page '{title}' does not exist."
    
    # Return the page summary
    summary = page.summary  # Short description of the page
    text = page.text  # Full page text (including sections)
    
    return summary, text

# Example: Fetch data for "Artificial intelligence"
title = "Artificial intelligence"
summary, full_text = fetch_wikipedia_page(title)

print("Summary:\n", summary)
print("\nFull Text:\n", full_text[:500])  # Printing the first 500 characters for readability


Summary:
 Artificial intelligence (AI) refers to the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals. Such machines may be called AIs.
High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., ChatGPT and AI art); and superhuman play and analysis in strategy games (e.g., chess and Go). However, many AI applications are not perceived as AI: "A lot of cutting edge AI has filtered into gener

In [11]:
import wikipediaapi
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
import os

# === Initialize Pinecone ===

# === Define index names for different domains ===
index_sports = "sports-embeddings"
index_music = "music-embeddings"

# === Create indexes if not already present ===
for index_name in [index_sports, index_music]:
    if index_name not in pc.list_indexes().names():
        pc.create_index(
            name=index_name,
            dimension=384,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )

# === Connect to each index ===
sports_index = pc.Index(index_sports)
music_index = pc.Index(index_music)

# === Wikipedia setup ===
wiki_wiki = wikipediaapi.Wikipedia(
    user_agent="MyProjectName (merlin@example.com)",
    language="en",
    extract_format=wikipediaapi.ExtractFormat.WIKI
)

# === Embedding model ===
model = SentenceTransformer('all-MiniLM-L6-v2')

# === Helper Functions ===
def fetch_wikipedia_page(title):
    page = wiki_wiki.page(title)
    if not page.exists():
        return None, None
    return page.summary, page.text

def chunk_text(text, chunk_size=300):
    words = text.split()
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

def generate_embeddings(chunks):
    return model.encode(chunks)

def store_in_index(index, chunks, embeddings, title):
    upsert_data = []
    for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
        vector_id = f"{title.replace(' ', '_')}_chunk_{i}"
        metadata = {"title": title, "chunk_index": i, "text": chunk}
        upsert_data.append((vector_id, embedding.tolist(), metadata))
    index.upsert(vectors=upsert_data)
    print(f"✅ Stored {len(chunks)} chunks for '{title}' in index.")

# === Store into respective index ===
def process_and_store(title, domain):
    summary, full_text = fetch_wikipedia_page(title)
    if not full_text:
        print(f"❌ Wikipedia page for '{title}' not found.")
        return
    chunks = chunk_text(full_text)
    embeddings = generate_embeddings(chunks)

    if domain == "sports":
        store_in_index(sports_index, chunks, embeddings, title)
    elif domain == "music":
        store_in_index(music_index, chunks, embeddings, title)
    else:
        print("❌ Unknown domain.")

# === Example Use ===
topic = input("🏷️ Enter Wikipedia topic: ")
domain = input("📁 Domain (sports / music): ").strip().lower()

process_and_store(topic, domain)


✅ Stored 36 chunks for 'football' in index.


In [ ]:
import requests
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
from uuid import uuid4

# === Groq setup ===
GROQ_API_KEY = "x"
GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"

# === Pinecone setup ===
pc = Pinecone(api_key="x")

# Define your domain-specific indexes
INDEX_MAP = {
    "sports": "sports-embeddings",
    "music": "music-embeddings"
}

# Create indexes if not exist
for index_name in INDEX_MAP.values():
    if index_name not in pc.list_indexes().names():
        pc.create_index(
            name=index_name,
            dimension=384,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )

# Connect to indexes
indexes = {domain: pc.Index(name) for domain, name in INDEX_MAP.items()}

# Embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# === Step 1: Classify query using LLM ===
def classify_topic(query):
    system_instruction = "You are a classifier. Only return the topic in one word. Do not explain."

    prompt = f"""
Classify the topic of this query into one of the following: music, sports.

Query: "{query}"
Topic:
""".strip()

    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": "llama3-8b-8192",
        "messages": [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0
    }

    response = requests.post(GROQ_API_URL, headers=headers, json=data)
    response.raise_for_status()
    result = response.json()
    
    # Extract and clean category
    raw_category = result["choices"][0]["message"]["content"]
    category = raw_category.strip().lower().split()[0]  # Take first word only

    print(f"🧠 Raw category response: {raw_category}")
    print(f"✅ Cleaned category: {category}")
    
    return category


# === Step 2: Query the correct index ===
def query_index(query, category):
    if category not in INDEX_MAP:
        return f"Sorry, no knowledge base available for: {category}"

    index = indexes[category]
    query_embedding = model.encode(query).tolist()

    results = index.query(
        vector=query_embedding,
        top_k=3,
        include_metadata=True
    )

    # Format results
    formatted = []
    for match in results.get("matches", []):
        meta = match["metadata"]
        formatted.append(f"- **{meta.get('title', 'Untitled')}**\n{meta.get('text', '')[:200]}...\n[Source]({meta.get('url', '#')})\n")

    return "\n".join(formatted) if formatted else "No relevant results found."

# === Step 3: Route the query ===
def route_query(query):
    category = classify_topic(query)
    print(f"\n🧠 Detected Category: {category.title()}")
    return query_index(query, category)

# === Run example ===
if __name__ == "__main__":
    user_query = input("Enter your query: ")
    response = route_query(user_query)
    print("\n📄 Top Results:\n")
    print(response)


🧠 Raw category response: music
✅ Cleaned category: music

🧠 Detected Category: Music

📄 Top Results:

- **songs**
popular music, a singer may perform with an acoustic guitarist, pianist, organist, accordionist, or a backing band. In jazz, a singer may perform with a single pianist, a small combo (such as a trio o...
[Source](#)

- **songs**
A song is a musical composition performed by the human voice. The voice often carries the melody (a series of distinct and fixed pitches) using patterns of sound and silence. Songs have a structure to...
[Source](#)

- **songs**
European art songs is considered as an important part of the composition. Some art songs are so revered that they take on characteristics of national identification. Art songs emerge from the traditio...
[Source](#)



In [ ]:
import requests
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
from uuid import uuid4

# === Groq setup ===
GROQ_API_KEY = "x"
GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"

# === Pinecone setup ===
pc = Pinecone(api_key="x")

# Define your domain-specific indexes
INDEX_MAP = {
    "sports": "sports-embeddings",
    "music": "music-embeddings"
}

# Create indexes if not exist
for index_name in INDEX_MAP.values():
    if index_name not in pc.list_indexes().names():
        pc.create_index(
            name=index_name,
            dimension=384,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )

# Connect to indexes
indexes = {domain: pc.Index(name) for domain, name in INDEX_MAP.items()}

# Embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# === Step 1: Classify query using LLM ===
def classify_topic(query):
    system_instruction = "You are a classifier. Only return the topic in one word. Do not explain."

    prompt = f"""
Classify the topic of this query into one of the following: music, sports.

Query: "{query}"
Topic:
""".strip()

    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": "llama3-8b-8192",
        "messages": [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0
    }

    response = requests.post(GROQ_API_URL, headers=headers, json=data)
    response.raise_for_status()
    result = response.json()
    
    # Extract and clean category
    raw_category = result["choices"][0]["message"]["content"]
    category = raw_category.strip().lower().split()[0]  # Take first word only

    print(f"🧠 Raw category response: {raw_category}")
    print(f"✅ Cleaned category: {category}")
    
    return category

# === Step 2: Query the correct index ===
def query_index(query, category):
    if category not in INDEX_MAP:
        return f"Sorry, no knowledge base available for: {category}"

    index = indexes[category]
    query_embedding = model.encode(query).tolist()

    results = index.query(
        vector=query_embedding,
        top_k=3,
        include_metadata=True
    )

    # Format results
    contexts = []
    for match in results.get("matches", []):
        meta = match["metadata"]
        text = meta.get('text', '')
        contexts.append(text)

    return contexts if contexts else ["No relevant context found."]

# === Step 3: Pass retrieved context + user query to LLM ===
def generate_answer(query, contexts):
    system_instruction = (
        "You are a helpful assistant. Use the provided context to answer the question accurately. "
        "If the context does not contain the answer, say 'Answer not found in context.'"
    )

    context_text = "\n\n".join(contexts)

    prompt = f"""
Context:
{context_text}

Question: {query}
Answer:
""".strip()

    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": "llama3-8b-8192",
        "messages": [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.2  # Slight creativity but mostly grounded
    }

    response = requests.post(GROQ_API_URL, headers=headers, json=data)
    response.raise_for_status()
    result = response.json()

    final_answer = result["choices"][0]["message"]["content"].strip()
    return final_answer

# === Step 4: Full pipeline ===
def route_query(query):
    category = classify_topic(query)
    print(f"\n🧠 Detected Category: {category.title()}")
    
    contexts = query_index(query, category)
    print(f"📚 Retrieved {len(contexts)} context documents.")

    answer = generate_answer(query, contexts)
    return answer

# === Run example ===
if __name__ == "__main__":
    user_query = input("Enter your query: ")
    response = route_query(user_query)
    print("\n📄 Final Answer:\n")
    print(response)


🧠 Raw category response: music
✅ Cleaned category: music

🧠 Detected Category: Music
📚 Retrieved 3 context documents.

📄 Final Answer:

According to the provided context, a song is a musical composition performed by the human voice. The voice often carries the melody (a series of distinct and fixed pitches) using patterns of sound and silence. Songs have a structure to them, such as the common ABA form, and are usually made of sections that are repeated or performed with variation later.
